In [1]:
#flint michigan water crisis notebook

# Install necessary libraries
!pip install folium geopandas


In [2]:
import folium
import geopandas as gpd
import requests
import pandas as pd

# Define your Census API key
CENSUS_API_KEY = "166bef5842180ce9dd54826ba95a44ebb1c2f877"

# Base URL for the Census API
BASE_URL = "https://api.census.gov/data/2020/acs/acs5"

# Variables to fetch
variables = {
    "B01003_001E": "Total Population",  # Total population
    "B02001_002E": "White Population",  # White alone
    "B02001_003E": "Black Population",  # Black or African American alone
    "B02001_005E": "Asian Population",  # Asian alone
    "B02001_006E": "Other Population",  # Other race
    "B02001_007E": "Two or More Races"  # Two or more races
}

# Counties of interest with FIPS codes
county_fips = {
    "Wayne":     "163",
    "Oakland":   "125",
    "Macomb":    "099",
    "Washtenaw": "161",
    "Monroe":    "115",
}

# Fetch demographic data for the counties
data = []
for county, code in county_fips.items():
    params = {
        "get": ",".join(variables.keys()),
        "for": f"county:{code}",
        "in": "state:26",  # Michigan FIPS code
        "key": CENSUS_API_KEY
    }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        rows = response.json()
        header = rows[0]
        values = rows[1]
        county_data = dict(zip(header, values))
        county_data["County"] = county
        data.append(county_data)

# Convert to a DataFrame
df = pd.DataFrame(data)
df.rename(columns=variables, inplace=True)

# Load counties GeoJSON directly from URL (no download needed)
url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
all_counties = gpd.read_file(url)

# Filter Michigan (STATEFP = 26)
michigan_counties = all_counties[all_counties["id"].str.startswith("26")]

# Filter for Detroit metro counties
detroit_counties = ['Wayne', 'Oakland', 'Macomb', 'Washtenaw', 'Monroe']
detroit_area = michigan_counties[michigan_counties['NAME'].isin(detroit_counties)]

# Merge demographic data with geographical data
merged = detroit_area.merge(df, left_on="NAME", right_on="County")

# Create a folium map centered on Detroit
detroit_map = folium.Map(location=[42.3314, -83.0458], zoom_start=9)

# Add counties with demographic information
for _, row in merged.iterrows():
    popup_text = (
        f"<b>County:</b> {row['NAME']}<br>"
        f"<b>Total Population:</b> {row['Total Population']}<br>"
        f"<b>White Population:</b> {row['White Population']}<br>"
        f"<b>Black Population:</b> {row['Black Population']}<br>"
        f"<b>Asian Population:</b> {row['Asian Population']}<br>"
        f"<b>Other Population:</b> {row['Other Population']}<br>"
        f"<b>Two or More Races:</b> {row['Two or More Races']}<br>"
    )
    folium.GeoJson(
        row['geometry'].__geo_interface__,
        name=row['NAME'],
        tooltip=folium.Tooltip(row['NAME']),
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(detroit_map)

# Display the map
detroit_map